In [1]:
# ======================================
# 1. Install Dependencies
# ======================================
!pip install datasets huggingface_hub

  Using cached xxhash-3.6.0-cp310-cp310-win_amd64.whl.metadata (13 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.4.1-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   --------------------------------------- 527.0/527.0 kB 16.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/625.2 kB ? eta -:--:--
   --------------------------------------- 625.2/625.2 kB 24.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 54.1 MB/s eta


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ======================================
# 2. Download Datasets
# ======================================
from datasets import load_dataset

# You can reduce size with split="train[:10%]"
math_ds = load_dataset("jeggers/competition_math", "original", split="train")
gsm8k_ds = load_dataset("gsm8k", "main", split="train[:20%]")

print("MATH sample:", math_ds[0])
print("GSM8K sample:", gsm8k_ds[0])

Generating test split: 100%|██████████| 5000/5000 [00:00<00:00, 56382.18 examples/s]
C:\workspace\math-reasoning-engine\.venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tonyt\.cache\huggingface\hub\datasets--gsm8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|████████

MATH sample: {'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.', 'extracted_solution': '0'}
GSM8K sample: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 

In [5]:
# ======================================
# 3. Normalize Format
# ======================================

def normalize_math(sample, idx):
    return {
        "id": f"math_{idx}",
        "problem": sample.get("problem", ""),
        "solution": sample.get("solution", ""),
        "source": "MATH"
    }


def normalize_gsm8k(sample, idx):
    return {
        "id": f"gsm8k_{idx}",
        "problem": sample.get("question", ""),
        "solution": sample.get("answer", ""),
        "source": "GSM8K"
    }

math_data = [normalize_math(x, i) for i, x in enumerate(math_ds)]
gsm8k_data = [normalize_gsm8k(x, i) for i, x in enumerate(gsm8k_ds)]

all_data = math_data + gsm8k_data

print("Total samples:", len(all_data))

Total samples: 8995


In [6]:
# ======================================
# 4. Cleaning
# ======================================
import re

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\\$.*?\\$", "", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

for item in all_data:
    item["problem"] = clean_text(item["problem"])
    item["solution"] = clean_text(item["solution"])

In [7]:
# ======================================
# 5. Step Extraction
# ======================================

def extract_steps(solution):
    if not solution:
        return []
    steps = solution.split("\n")
    return [s.strip() for s in steps if len(s.strip()) > 10]


# ======================================
# 6. Theorem Extraction (Simple Rule-Based)
# ======================================
THEOREMS = ["AM-GM", "Cauchy", "Pigeonhole", "Jensen", "Triangle inequality"]

def extract_theorems(text):
    found = []
    for t in THEOREMS:
        if t.lower() in text.lower():
            found.append(t)
    return found

In [8]:
# ======================================
# 7. Feature Engineering
# ======================================

for item in all_data:
    item["steps"] = extract_steps(item["solution"])
    item["theorems"] = extract_theorems(item["solution"])
    item["difficulty"] = len(item["steps"]) + len(item["theorems"])

In [9]:
# ======================================
# 8. Split Task vs Knowledge
# ======================================

task_data = []
knowledge_data = []

for item in all_data:
    if item["solution"] and len(item["steps"]) > 2:
        knowledge_data.append(item)
    else:
        task_data.append(item)

print("Task size:", len(task_data))
print("Knowledge size:", len(knowledge_data))

Task size: 4670
Knowledge size: 4325


In [14]:
# ======================================
# 9. Save to JSONL
# ======================================
import json
import os

os.makedirs("../data", exist_ok=True)


def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl(all_data, "../data/unified.jsonl")
save_jsonl(task_data, "../data/task.jsonl")
save_jsonl(knowledge_data, "../data/knowledge.jsonl")

print("Saved all datasets!")

Saved all datasets!


In [11]:
# ======================================
# 10. Quick Inspection
# ======================================

import random
sample = random.choice(knowledge_data)

print("\n=== SAMPLE ===")
print("Problem:\n", sample["problem"])
print("\nSteps:\n", sample["steps"][:3])
print("\nTheorems:\n", sample["theorems"])


=== SAMPLE ===
Problem:
 In $\triangle{ABC}$, $\angle ABC=120^\circ,AB=3$ and $BC=4$. If perpendiculars constructed to $\overline{AB}$ at $A$ and to $\overline{BC}$ at $C$ meet at $D$, then $CD=$
$\text{(A) } 3\quad \text{(B) } \frac{8}{ qrt{3}}\quad \text{(C) } 5\quad \text{(D) } \frac{11}{2}\quad \text{(E) } \frac{10}{ qrt{3}}$

Steps:
 ['We begin by drawing a diagram.[asy] import olympiad; import cse5; import geometry; size(150); defaultpen(fontsize(10pt)); defaultpen(0.8); dotfactor = 4; pair A = origin; pair C = A+dir(55); pair D = A+dir(0); pair B = extension(A,A+dir(90),C,C+dir(-155)); label("$A$",A,S); label("$C$",C,NE); label("$D$",D,SE); label("$B$",B,NW); label("$4$",B--C,NW); label("$3$",A--B,W); draw(A--C--D--cycle); draw(A--B--C); draw(rightanglemark(B,C,D,2)); draw(rightanglemark(B,A,D,2)); [/asy]We extend $CB$ and $DA$ to meet at $E.$ This gives us a couple right triangles in $CED$ and $BEA.$[asy] import olympiad; import cse5; import geometry; size(250); defaultpen(fon